<h2>
Environment
</h2>

<p> 
Python 3.13.16,
pip 26.2.1,
Random Seed = 11,
</P>


In [24]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

<h2>Data Audit (Raw)</h2>

<p>
Below you can see our data sets shape, column names, data types and first 5 data entries. As each passanger is given a passenger is given a unique passangerId it is highly unlikely for there to be any duplicate data. However there is a slight exception in that of SibSp(int64 number of siblings or spouses aboard) and Parch(int64 number of parents or children aboard) due to their inherinetly grouped values (without an explicit link like family group etc.) a SibSp/Parch group or pair can be split between training and testing sets leading to data leaked if indeed there is a high correelatoin between SibSp/Parch number and survival rate. We will fruther explain and combat this issue later on in this notebook.

Pclass which represents passenger ticket class where 1, 2 and 3 represent 1st, 2nd and 3rd respectively and embarked C, Q and S which represent Cherbourg, Queenstown and Southampton respectively. Finally, Cabin string has a letter and number pair representing the deck A,B,C,D or E and their respective rooms for example A1 through A37 wihout A13 for supersticious reasons for a total of 36 rooms. B, C, D and E had 101, 134, 49 and 41 rooms respectively with E also having first class rooms E-200, E-201, E-202 and E-203. Altough the extensive knowlege of the room numbers and layout is not very useful to our model as 77% of this data is NULL meaning we will more than likely have to drop this table. Apart from these outliers the remaining columns (PassengerId, Survived, Name, Sex, Age, Ticket and fare)have very self explanitory data types which can be seen below for further inspection.

Apart from the affor mentioned missing cabin data there are also small amounts of missing data in age and embarked. Columns irelevent to survival like ticket number will likely be dropped from training however, age is a very useful survival indicator so an imputation scheme for these missing values must be decided. Likely a mean imputation overall, based on gender, name (MR, MRS or MS), or a mixture of these to get the most accurate possible data for these missing values.
</p>

In [14]:
# Load titanic dataset (CSV file)
titanic_data = pd.read_csv('Titanic-Dataset.csv')

print(titanic_data.shape)
print(titanic_data.columns)
print(titanic_data.dtypes)

titanic_data.head()

(891, 12)
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
X = titanic_data.drop(columns=["Survived", "Cabin"]) # drop target variable and Cabin column (too many missing values)
y = titanic_data["Survived"] # target variable

model = LogisticRegression(max_iter=1000) # chosen model for classification task

class GroupMeanImputer(BaseEstimator, TransformerMixin): # custom impputer to fill missing values in a age column based on the mean of a grouping column (Sex in this case)
    def __init__(self, group_col="Sex", target_col="Age"):
        self.group_col = group_col
        self.target_col = target_col

    def fit(self, X, y=None):
        self.means_ = X.groupby(self.group_col)[self.target_col].mean()
        return self

    def transform(self, X):
        X = X.copy()
        X[self.target_col] = X[self.target_col].fillna(
            X[self.group_col].map(self.means_)
        )
        return X


strata = (
    titanic_data["Pclass"].astype(str) + "_"
    + titanic_data["Sex"] + "_"
    + titanic_data["Survived"].astype(str)
)

X_train, X_test, y_train, y_test = train_test_split( #20% of the data is used for testing, and the rest for training. Stratified sampling is used to ensure that the distribution of classes in the target variable is similar in both the training and testing sets.
    X, y, test_size=0.2, stratify=strata, random_state=11
)

categorical = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")), # simple imputer is used to fill missing values in categorical columns with the most frequent value in that column (for us embarked and sex columns as in preprocessor below)
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([ 
    ("categorical", categorical, ["Sex", "Embarked"]),
    ("numeric", StandardScaler(), ["Pclass", "Age", "SibSp", "Parch", "Fare"]),
])  # PassengerId Ticket and Name are dropped here by default

pipe = Pipeline([ # apply above transformations and model in a pipeline
    ("age_impute", GroupMeanImputer(group_col="Sex", target_col="Age")),
    ("preprocess", preprocessor),
    ("model", model),
])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=11)
scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
print(f"{scores.mean():.3f} +/- {scores.std():.3f}") # non final simply to prove that code is working with a model (regression in this case)and that the pipeline is working correctly.


0.800 +/- 0.022


<h2>Preprocessig Pipeline and data audit</h2>

<p>
Survived (our targer variable) and cabin (due to high numbers of missing data) are dropped.
For the missing categorical values such as embarked we use a simple imputer with most frequent values for our strategy
For the missing values in age we create a custom imputor which gets the mean value based on the sex rather than an overall mean.
Standerd numeric scailing is applied to Pclass, age, sibsp, parch and Fare

PassengerId, Ticket and Name are left out by the model but not dropped outright. This is due to their relative unusefullness in training our model however, as mentioned earlier our Parch and SibSp columns add a level of complexity to the model. Ticket, Name and Fare can possibly help us create a more accurate family group later on as a comparison to our base model as many familys, spouses or other such groups often share tickets or last names. Fare can aditionally be used as a cross reference for shared tickets. Additionally name column contains titles such as MR, MRS, MS or Master meaning we can create an even more accurate custom age imputer if we find our one inaccurate in future.

I have chosen a 20% test split and a stratified K fold cross validation as data is limited with mild class imbalance and we want the model to be as accurate as possible with unseen data.

These decisions give me the most accurate baseline model with clear headroom for improvement particularly around the Parch SibSp conundrum (major reason as to why I am keeping so many columns even if they seem useless to my model in its current state)
</p>

In [ ]:
dummymodel = DummyClassifier(strategy="most_frequent") # baseline model to compare against
dummy_scores = cross_val_score(dummymodel, X_train, y_train, cv=cv, scoring="accuracy")
print(f"Dummy model accuracy: {dummy_scores.mean():.3f} +/- {dummy_scores.std():.3f}")

sex_predictions = (X_train["Sex"] == "female").astype(int) #prediction based on sex, where female is predicted to survive and male is predicted to not survive as sex is the strongest column exclusve predictor of survival in the Titanic dataset

print(f"Accuracy of predicting survival based on sex: {accuracy_score(y_train, sex_predictions):.3f}")

Dummy model accuracy: 0.615 +/- 0.003
Accuracy of predicting survival based on sex: 0.787
